### Project 3: Fine-Tuning an LLM with Low Rank Adaptation (LoRA).

In this project, you are asked to fine-tune and evaluate a transformer-based
language model called ELECTRA (Efficiently Learning an Encoder that Classifies
Token Replacements Accurately) to classify online customer service queries.

ELECTRA is a BERT-sized language model developed by Google Research that learns
language more efficiently than traditional masked language models by training
on a richer, token-level discrimination task. The dataset you will work with,
BANKING77, contains over 13,000 short customer service messages drawn from the
online banking domain — the kind of text a user might type into a bank's
support chatbot — each labeled with one of 77 fine-grained intent categories
such as identifying a failed transaction, requesting a replacement card, or
querying an exchange rate. The queries are short, realistic, and often subtly different
from one another, making accurate classification a genuine challenge for any
model.

Rather than training ELECTRA from scratch, you will apply LoRA (Low-Rank
Adaptation), a parameter-efficient fine-tuning technique that updates only a
small number of additional weights while keeping the base model largely frozen
— significantly reducing the compute and memory costs of fine-tuning. You will
systematically explore the effect of key hyperparameters on model performance,
and track your experiments using Weights & Biases (W&B), a professional machine
learning experiment tracking platform. W&B automatically logs training metrics
such as loss and accuracy in real time, and provides an interactive dashboard
where you can visualize learning curves, compare runs across different
hyperparameter configurations, and identify where and why model performance
changes. You will use W&B to help interpret your training results and to
draw meaningful conclusions about model behavior. Finally,
you will present your findings in a professional format suitable for a
technical audience.

By the end of this project, you will have hands-on
experience with the full fine-tuning pipeline: from data preparation and
efficient adaptation of a pretrained transformer, through experimentation and
performance analysis, to clear and professional communication of results.

#### Resources

- PEFT Documentation: https://huggingface.co/docs/peft/
- Banking77 Paper: https://arxiv.org/abs/2003.04807
- ELECTRA Paper: https://arxiv.org/abs/2003.10555
- Wandb Tutorial: https://docs.wandb.ai/quickstart

In [2]:
#### Setup and Installation - Colab

# Uncomment and run if using Google Colab
!pip install -U transformers datasets peft wandb scikit-learn
!pip install -U torch torchvision torchaudio
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: wandb
    Found existing installation: wandb 0.26.1
    Uninstalling wandb-0.26.1:
      Successfully uninstalled wandb-0.26.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
     

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0



Additional Notes:

If you are running on Colab, enable the GPU runtime via Runtime -> Change runtime type -> T4 GPU (or similar)

If you are running the model locally, create & activate a virtual environment using the requirements.txt file
included with the assignment materials.


In [1]:
#### SECTION 1: IMPORTS

import wandb
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import (classification_report,confusion_matrix)

from google.colab import userdata


W0520 18:30:19.348000 10282 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0520 18:30:19.434000 10282 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
# For Local Computer:  Set Device and Confirm GPU or CPU
# Comment this section out if using Colab

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print('Using device:', DEVICE)
print('PyTorch version:', torch.__version__)




Using device: cuda
PyTorch version: 2.12.0+cu130


### SECTION 2: CONFIGURATION DICTIONARY

In [4]:


# TODO: you will need to input your selections here. You may find it helpful to review published articles and model descriptions to identify your starting points.

CONFIG = {
    "model"         : "google/electra-base-discriminator",

    "batch_size"    : 16,     # TODO: range 8-32, Electra is sensitive to small batches

    "learning_rate" : 2e-4,   # TODO: range 1e-4 to 5e-4. Avoid going below 1e-5 or above 1e-3.
                               # Increased from 2e-5 because validation loss was still steadily decreasing
                               # and the model appears to be underfitting rather than overfitting.

    "epochs"        : 10,     # TODO: range 1-10
                               # Increased because validation metrics were still improving at epoch 8.

    "lora_r"        : 16,     # TODO: range 4-16, 4 = minimal params/fast; 8 = balanced;
                               # 16 = richer but risks overfitting on small data.
                               # Increased adapter capacity to improve intent separation.

    "lora_alpha"    : 32,     # TODO: range 8-32. Convention is alpha = 2*lora_r

    "lora_dropout"  : 0.05,   # TODO: range 0.0-0.1 applied to adapters only.
                               # Keeping moderate dropout since overfitting was minimal.

    "max_length"    : 128,    # TODO: range 64-512. Memory costs increase with length.
                               # Reduced because Banking77 queries are short and rarely require 250 tokens.

    "seed"          : 42,

    "target_modules": ["query", "key", "value"],
                               # LoRA adapters applied to transformer attention projection matrices.

    "wandb_project" : "llm-hw3", # you will use this to identify your project in Weights and Biases.
}

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

#### SECTION 3: WANDB SETUP FUNCTION

In [5]:
def setup_wandb(config: dict) -> None:
    """
    Authenticate with WandB and initialise a new experiment run.

    wandb.login() checks for cached credentials first. If none are found
    it prompts interactively — you only need to do this once.

    wandb.init() opens the run and logs the full CONFIG dict as the
    hyperparameter record for this experiment.
    """
    print("Connecting to WandB...")
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    wandb.init(
        project = config["wandb_project"],
        name    = (
            f"electra-lora"
            f"_r{config['lora_r']}"
            f"_lr{config['learning_rate']}"
            f"_ep{config['epochs']}"
        ),
        config  = config,
    )
    print(f"  WandB run     : {wandb.run.name}")
    print(f"  Dashboard URL : {wandb.run.url}\n")

### SECTION 4: DATA LOADER FUNCTION

TODO:

    Load the raw Banking77 dataset:

    Create these reference variables:
        - dataset
        - label_names (list[str])
        - num_labels (int)
        
    Print the following in f-string format:
        - raw number of Train data
        - raw number of Test data
        - number of labels
        - sample of label_names (intent categories)
    
    Return {dict}
        Keys -> train, test, num_labels, label_names
        


In [6]:

def load_data() -> dict:
    """
    Load the raw Banking77 dataset and print a brief summary.

    Returns a dict with keys:
        train       — raw training split
        test        — raw test split
        num_labels  — number of intent categories (int)
        label_names — ordered list of intent category names (list[str])
    """

    print("Loading Banking77 dataset...")
    # load the dataset
    dataset = load_dataset("banking77")

    # extract label names
    label_names = dataset['train'].features['label'].names

    # extract number of labels
    num_labels  = len(dataset['train'].features['label'].names)

    # print summary
    print("Train examples    :", len(dataset["train"]))
    print("Test examples     :", len(dataset["test"]))
    print("Intent categories :", num_labels)
    print("Sample intents    :", label_names[:10])

    # return dictionary
    return {
        "train": dataset["train"],
        "test": dataset["test"],
        "num_labels": num_labels,
        "label_names": label_names,
    }

### SECTION 5: LOAD TOKENIZER FUNCTION

In [7]:
def load_tokenizer(config: dict) -> AutoTokenizer:
    """
    Load the tokenizer for the configured model.
    Ensures a pad token is set, which is required for batch processing.

    Returns the configured tokenizer.
    """
    # load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config['model'])

    # ensuring padding is set, if not set to eos
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer

### SECTION 6:TOKENIZATION FUNCTION

In [8]:
def tokenize_data(raw_data: dict, tokenizer: AutoTokenizer, config: dict) -> dict:
    """
    Tokenize the train and test splits, then divide the training data
    into train and validation sets.

    Sequences are truncated to max_length but NOT padded here.
    Padding is handled dynamically per batch by DataCollatorWithPadding,
    which the Trainer constructs automatically from processing_class=tokenizer.

    Returns a dict with keys:
        train — tokenized training split
        val   — tokenized validation split (held out from train)
        test  — tokenized test split
    """
    # Complete the tokenize helper below.
    #      - Truncate sequences longer than max_length.



    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=config["max_length"],
        )


    # Complete the following:
        # - use map() to tokenize the raw data for train and test
        # - for train and test include attributes: batched=True, remove_columns=["text"]

    tokenized_train = raw_data["train"].map(
        tokenize,
        batched=True,
        remove_columns=["text"],
    )

    tokenized_test = raw_data["test"].map(
        tokenize,
        batched=True,
        remove_columns=["text"],
    )

    splits = tokenized_train.train_test_split(
        test_size=0.2,
        seed=config["seed"],
    )

    print(
        f"Split sizes — "
        f"Train: {len(splits['train']):,} | "
        f"Val: {len(splits['test']):,} | "
        f"Test: {len(tokenized_test):,}\n"
    )

    # use splits variable to return updated train and val data
    return {
        "train": splits["train"],
        "val": splits["test"],
        "test": tokenized_test,
    }

### SECTION 7: EVALUTATION METRICS

This is a Trainer callback.

In [9]:
def compute_metrics(eval_pred) -> dict:
    """
    Evaluation callback passed to the Trainer.
    Converts raw logits to class predictions and computes accuracy and
    weighted precision, recall, and F1 via a single classification_report call.
    Called automatically after each evaluation step — and by the Trainer's
    WandB integration, which picks up whatever dict this function returns and
    logs it to the dashboard automatically.
    """
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=1)
    report         = classification_report(labels, predictions, output_dict  = True, zero_division= 0)
    weighted       = report["weighted avg"]

    return {
        "accuracy"           : report["accuracy"],
        "f1_weighted"        : weighted["f1-score"],
        "precision_weighted" : weighted["precision"],
        "recall_weighted"    : weighted["recall"],
    }


### SECTION 8: MODEL EVALUATION

In [10]:
def evaluate_model(model, dataset, tokenizer: AutoTokenizer, config: dict) -> dict:
    """
    Run inference on a dataset and return metrics and raw predictions.
    This function is reused identically for the baseline and all post-training
    evaluations — the model passed in is the only thing that changes.

    report_to is intentionally kept empty here because this function is used
    for both the pre-training baseline (before the WandB run opens) and for
    final evaluation. Metrics from final evaluation are logged explicitly
    via wandb.summary in Section 12.

    Returns a dict with keys:
        accuracy    — float
        f1_weighted — float
        predictions — np.ndarray of predicted class indices
        labels      — np.ndarray of true class indices
    """
    eval_trainer = Trainer(
        model            = model,
        args             = TrainingArguments(
            output_dir                 = "./tmp_eval",
            per_device_eval_batch_size = config["batch_size"],
            dataloader_pin_memory      = False,
            remove_unused_columns      = False,
            report_to                  = [],
        ),
        processing_class = tokenizer,
        compute_metrics  = compute_metrics,
    )

    output = eval_trainer.predict(dataset)

    return {
        "accuracy"   : output.metrics["test_accuracy"],
        "f1_weighted": output.metrics["test_f1_weighted"],
        "predictions": np.argmax(output.predictions, axis=1),
        "labels"     : output.label_ids,
    }

### SECTION 9: LORA MODEL SETUP FUNCTION

Some things to consider

(a) Why do we target "query", "key", and "value" specifically,rather than all layers in the model?
(b) Why is bias="none" the standard choice for LoRA?
(c) What is the relationship between lora_r and lora_alpha, and what happens if you set alpha much higher than 2*r?

In [11]:

def create_lora_model(model, config: dict) -> tuple:
    """
    Wrap the base model with LoRA adapters and report parameter efficiency.
    Only the adapter weights are trainable; the original model weights are frozen.

    Returns:
        lora_model — the PEFT-wrapped model
        param_pct  — percentage of parameters that are trainable (float)
    """
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        target_modules=config["target_modules"],
        bias="none",
        lora_dropout=config['lora_dropout'],
    )
    lora_model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in lora_model.parameters())
    param_pct = 100 * trainable / total

    print("LoRA adapters applied:")
    print(f"  Target modules   : {list(lora_config.target_modules)}")
    print(f"  Rank (r)         : {lora_config.r}")
    print(f"  Alpha            : {lora_config.lora_alpha}")
    print(f"  Trainable params : {trainable:,}  ({param_pct:.3f}% of {total:,} total)\n")

    return lora_model, param_pct


### SECTION 10: TRAINING ARGUMENTS FUNCTION

In [12]:
# Detect the current hardware at runtime and returns the correct precision settings as a dictionary, so you don't have to hardcode them manually.

def get_precision_flags():
    # Colab / Windows / Linux — CUDA GPU available
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return {"bf16": True,  "fp16": False}  # Modern NVIDIA (A100, L4)
        else:
            return {"bf16": False, "fp16": True}   # Older NVIDIA (T4, V100)

    # macOS Apple Silicon — MPS available
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return {"bf16": False, "fp16": False}       # MPS: neither supported reliably

    # CPU fallback (Windows no-GPU, older Macs)
    else:
        return {"bf16": False, "fp16": False}

precision = get_precision_flags()

In [13]:
def build_training_args(config: dict) -> TrainingArguments:
    """
    Declare all training hyperparameters in one place.
    Keeping this separate from the training loop makes it straightforward to
    experiment with different configurations without touching the training logic.

    Returns a configured TrainingArguments object.
    """
    return TrainingArguments(
        output_dir="./lora_banking_model",

        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],

        learning_rate=config["learning_rate"],

        max_grad_norm=1.0,
        warmup_ratio=0.06,
        weight_decay=0.01,

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,

        load_best_model_at_end=True,
        metric_for_best_model="f1_weighted",
        greater_is_better=True,

        **precision,

        dataloader_pin_memory=False,
        remove_unused_columns=False,

        seed=config["seed"],
        report_to=["wandb"],
    )

### SECTION 11: TRAINING LOOP FUNCTION

In [14]:
def run_training(
    model,
    data: dict,
    tokenizer: AutoTokenizer,
    training_args: TrainingArguments,
) -> Trainer:
    """
    Instantiate the Trainer with the prepared data and run the fine-tuning loop.
    The best checkpoint (by f1_weighted on the validation set) is reloaded
    automatically when training completes.

    Returns the trained Trainer instance.
    """
    trainer = Trainer(
        model            = model,
        args             = training_args,
        train_dataset    = data["train"],
        eval_dataset     = data["val"],
        processing_class = tokenizer,
        compute_metrics  = compute_metrics,
    )

    print("Starting LoRA fine-tuning...")
    trainer.train()
    print("Training complete.\n")

    return trainer

### SECTION 12: ERROR ANALYSIS FUNCTION

In [15]:
def analyze_errors(predictions, labels, label_names: list) -> dict:
    """
    Surface two complementary views of model failure on the validation set.

    VIEW 1 — Confusion pairs (from the confusion matrix)
        Shows which specific intent pairs are most often mixed up.
        Answers: "Which two intents does the model confuse most?"
        This is useful for spotting localised, pairwise errors.

    VIEW 2 — Per-class precision, recall, and F1 (from classification_report)
        Shows where the model is weakest at the individual intent level.
        Precision answers: "When the model predicts intent X, how often is
            it correct?" — low precision means over-prediction of that intent.
        Recall answers: "Of all queries that are intent X, how many does the
            model catch?" — low recall means the model is missing that intent.
        F1 is the harmonic mean of both and is the primary sort key here.
        Together, precision and recall reveal whether a weak intent is caused
        by over-triggering, under-triggering, or both — a distinction that
        aggregate metrics like weighted F1 hide entirely.

    Returns a dict with keys:
        confusion_pairs     — list of (true_label, predicted_label, count) tuples,
                              sorted by count descending
        per_class           — list of (label, precision, recall, f1, support) tuples,
                              sorted by f1 ascending (weakest intents first)
        accuracy            — float, overall accuracy
        f1_weighted         — float, weighted F1
        precision_weighted  — float, weighted precision
        recall_weighted     — float, weighted recall
    """
    # --- View 1: confusion pairs ---
    cm    = confusion_matrix(labels, predictions)
    pairs = [
        (label_names[i], label_names[j], cm[i, j])
        for i in range(len(label_names))
        for j in range(len(label_names))
        if i != j and cm[i, j] > 0
    ]
    pairs.sort(key=lambda x: x[2], reverse=True)

    print("Top 10 Most Confused Intent Pairs (Validation Set):")
    print(f"{'True Intent':<36} {'Predicted As':<36} {'Count':>5}")
    print("-" * 79)
    for true_label, pred_label, count in pairs[:10]:
        print(f"{true_label:<36} {pred_label:<36} {count:>5}")

    # --- View 2: per-class precision, recall, F1 ---
    report   = classification_report(
        labels,
        predictions,
        target_names = label_names,
        output_dict  = True,
        zero_division= 0,
    )
    per_class = [
        (
            label,
            report[label]["precision"],
            report[label]["recall"],
            report[label]["f1-score"],
            int(report[label]["support"]),
        )
        for label in label_names
    ]
    per_class.sort(key=lambda x: x[3])   # ascending by F1 — weakest first

    print(f"\n10 Weakest Intents by F1 (Validation Set):")
    print(f"{'Intent':<36} {'Precision':>10} {'Recall':>8} {'F1':>8} {'N':>5}")
    print("-" * 69)
    for label, prec, rec, f1, n in per_class[:10]:
        print(f"{label:<36} {prec:>10.3f} {rec:>8.3f} {f1:>8.3f} {n:>5}")

    weighted = report["weighted avg"]
    return {
        "confusion_pairs"    : pairs,
        "per_class"          : per_class,
        "accuracy"           : report["accuracy"],
        "f1_weighted"        : weighted["f1-score"],
        "precision_weighted" : weighted["precision"],
        "recall_weighted"    : weighted["recall"],
    }

### SECTION 13: MAIN EXECUTION

In [16]:

# Step 1 — Connect to WandB
#   This must happen before training so the run is open when the
#   Trainer starts streaming metrics. The baseline is evaluated
#   before wandb.init() on purpose: it uses a bare Trainer with
#   report_to=[] (see evaluate_model), so no run needs to be open yet.

setup_wandb(CONFIG)

# Step 2 — Load and tokenize data

raw_data  = load_data()
tokenizer = load_tokenizer(CONFIG)
data      = tokenize_data(raw_data, tokenizer, CONFIG)

# Step 3 — Load base model and establish a pre-training baseline

print("Loading base ELECTRA model...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model"],
    num_labels=raw_data["num_labels"],
).to(DEVICE)

print("Evaluating baseline (untrained) performance...")
baseline = evaluate_model(base_model, data["val"], tokenizer, CONFIG)
print(f"  Baseline Accuracy    : {baseline['accuracy']:.4f}")
print(f"  Baseline F1-Weighted : {baseline['f1_weighted']:.4f}\n")

# Log baseline to WandB so it appears alongside training metrics in the
# run summary. wandb.summary values are pinned to the run — they don't
# appear as time-series points but are always visible in the run overview.

wandb.summary["baseline_accuracy"]    = baseline["accuracy"]
wandb.summary["baseline_f1_weighted"] = baseline["f1_weighted"]

# Step 4 — Apply LoRA adapters to the base model

lora_model, param_pct = create_lora_model(base_model, CONFIG)
wandb.summary["trainable_param_pct"] = param_pct

# Step 5 — Configure and run fine-tuning
#   From this point onward the Trainer streams loss, learning rate, and
#   eval metrics to WandB automatically via report_to=["wandb"].

training_args = build_training_args(CONFIG)
trainer       = run_training(lora_model, data, tokenizer, training_args)

# Step 6 — Evaluate the fine-tuned model on validation and test sets

val_results  = evaluate_model(trainer.model, data["val"],  tokenizer, CONFIG)
test_results = evaluate_model(trainer.model, data["test"], tokenizer, CONFIG)

# Print the comparison table to the console
print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"\n{'Metric':<22} {'Baseline':>10} {'Val (LoRA)':>12} {'Test (LoRA)':>12}")
print("-" * 58)
print(f"{'Accuracy':<22} {baseline['accuracy']:>10.4f} {val_results['accuracy']:>12.4f} {test_results['accuracy']:>12.4f}")
print(f"{'F1-Weighted':<22} {baseline['f1_weighted']:>10.4f} {val_results['f1_weighted']:>12.4f} {test_results['f1_weighted']:>12.4f}")
print(f"\n  Accuracy gain vs baseline : {val_results['accuracy'] - baseline['accuracy']:+.4f}")

gap  = val_results["accuracy"] - test_results["accuracy"]
flag = "✓ Good generalization" if abs(gap) < 0.02 else "⚠ Check for overfitting"
print(f"  Val → Test gap            : {gap:+.4f}  ({flag})")

# Log final evaluation metrics and the generalization gap to WandB summary.
# These join the baseline values logged earlier, giving a complete
# before/after picture on the run's overview page.

wandb.summary["val_accuracy"]         = val_results["accuracy"]
wandb.summary["val_f1_weighted"]      = val_results["f1_weighted"]
wandb.summary["test_accuracy"]        = test_results["accuracy"]
wandb.summary["test_f1_weighted"]     = test_results["f1_weighted"]
wandb.summary["accuracy_gain"]        = val_results["accuracy"] - baseline["accuracy"]
wandb.summary["val_test_gap"]         = gap

# Step 7 — Analyze the most common misclassifications

print()
error_analysis  = analyze_errors(
    val_results["predictions"],
    val_results["labels"],
    raw_data["label_names"],
)

# Log the single weakest intent to WandB summary as a concrete signal of
# where the model needs the most improvement.

weakest_label, weakest_prec, weakest_rec, weakest_f1, _ = error_analysis["per_class"][0]
wandb.summary["weakest_intent"]           = weakest_label
wandb.summary["weakest_intent_precision"] = weakest_prec
wandb.summary["weakest_intent_recall"]    = weakest_rec
wandb.summary["weakest_intent_f1"]        = weakest_f1

# Step 8 — Close the WandB run cleanly
#   wandb.finish() flushes any remaining buffered metrics, marks the run
#   as complete in the dashboard, and releases the connection. Without
#   this call the run stays in a "running" state until it times out.

wandb.finish()
print("WandB run closed.")

Connecting to WandB...


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chris-r-thompson1212 (chris-r-thompson1212-university-of-denver) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  WandB run     : electra-lora_r16_lr0.0002_ep10
  Dashboard URL : https://wandb.ai/chris-r-thompson1212-university-of-denver/llm-hw3/runs/d1lu4htu

Loading Banking77 dataset...
Train examples    : 10003
Test examples     : 3080
Intent categories : 77
Sample intents    : ['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire']
Split sizes — Train: 8,002 | Val: 2,001 | Test: 3,080

Loading base ELECTRA model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings_project.bias                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings_project.weight                 | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,Precision Weighted,Recall Weighted
1,2.598496,2.230788,0.401299,0.335768,0.449131,0.401299
2,1.087906,0.971249,0.733633,0.715888,0.750959,0.733633
3,0.753700,0.613424,0.822589,0.820950,0.838558,0.822589
4,0.649170,0.503549,0.861569,0.860368,0.868735,0.861569
5,0.445773,0.473173,0.865067,0.863568,0.874006,0.865067
6,0.369949,0.438333,0.888556,0.888116,0.893731,0.888556
7,0.355461,0.405014,0.894053,0.894230,0.901145,0.894053
8,0.309457,0.402173,0.890055,0.889978,0.895323,0.890055
9,0.241648,0.394240,0.898051,0.898154,0.902735,0.898051
10,0.247225,0.387963,0.899050,0.898964,0.902527,0.899050


Training complete.



FINAL RESULTS

Metric                   Baseline   Val (LoRA)  Test (LoRA)
----------------------------------------------------------
Accuracy                   0.0075       0.8991       0.8964
F1-Weighted                0.0009       0.8990       0.8961

  Accuracy gain vs baseline : +0.8916
  Val → Test gap            : +0.0026  (✓ Good generalization)

Top 10 Most Confused Intent Pairs (Validation Set):
True Intent                          Predicted As                         Count
-------------------------------------------------------------------------------
declined_card_payment                card_not_working                         4
direct_debit_payment_not_recognised  card_payment_not_recognised              4
transfer_not_received_by_recipient   pending_transfer                         4
balance_not_updated_after_bank_transfer transfer_not_received_by_recipient       3
beneficiary_not_allowed              failed_transfer                          3
extra_charge_on_statement   

eval/accuracy,▁▆▇▇██████
eval/f1_weighted,▁▆▇███████
eval/loss,█▃▂▁▁▁▁▁▁▁
eval/precision_weighted,▁▆▇▇██████
eval/recall_weighted,▁▆▇▇██████
eval/runtime,▁▅█▆▆██▆▇▆
eval/samples_per_second,█▄▁▃▃▁▁▂▂▃
eval/steps_per_second,█▄▁▃▃▁▁▂▂▃
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇███
+3,...


WandB run closed.
